# **STAGE 1 - ANONYMISE**

## **Objectives**

* In this notebook, I will be creating an anonymised version of the raw data - this will be the dataset analysed and transformed in **Stage 2 - ETL**.

> *Ethical Considerations*:
<br><br>When handling this data, I made the decision to create a separate Jupyter Notebook with the purpose of anonymising the raw data.
<br><br>The raw data contains **PII (Personally Identifiable Information)** in the form of a bank client number - `CLIENTNUM` - which is a unique identifier for each client of the bank. Given the sensitive nature of the data I am analysing - financial information - it is necessary to ensure this information cannot be traced back to an individual.
<br><br>The raw data I am manipulating in this Jupyter Notebook will therefore be excluded from my repository in GitHub and will be included in `.gitignore`.
<br><br>

## **Inputs**

* The data input I need for this section is taken from Kaggle: `bank-churners-raw.csv`.
* It will be saved in a folder called `datasets/raw-data/`.

## **Outputs**

* The output of **Stage 1 - Anonymise** will be an anonymised version of the raw dataset.
* You will see at the end of this Jupyter Notebook that the column `CLIENTNUM` will be anonymised using salting and hashing techniques. This column will then be dropped and replaced with one called `anonymised_clientnum`.

## **Additional Comments**

* I have separated this anonymisation stage from the ETL stage and have placed it in a separate Jupyter Notebook. 
* The rationale behind this decision is that **no code cells in this Jupyter Notebook will be runnable** because I am *excluding* the raw data from my GitHub upload. 
* The second section - **Stage 2 - ETL** - will be runnable because I will include the anonymised raw dataset in my repository.

---

## **Change Working Directory**

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis'

---

## **Import Libraries and Packages**

In order to anonymise the raw dataset, I need to import the following packages and libraries:

In [4]:
import pandas as pd
import hashlib
from dotenv import load_dotenv

---

## **1. Extract Raw Data**

### **1.1 Exclude from Repository Updates**

Prior to loading the raw dataset, there were some key steps I needed to take to ensure that the raw data and its PII would be excluded from uploads to GitHub.
* The first thing I did was create a folder in my `datasets` folder: `datasets/raw-datasets/`.
* I then ensured the relative path for this folder was placed in the `.gitignore` file.

### **1.2 Load Dataset**

Now that the above actions have been taken, I can move on with anonymising the data in this notebook. Firstly, I am going to read the raw dataset -`bank-churners-raw.csv` - into a DataFrame and will analyse which columns need handling.
* Using `print(df.columns)`, I can see that the column `CLIENTNUM` needs to be anonymised.

In [5]:
# Load raw dataset
df = pd.read_csv("datasets/raw-data/bank-churners-raw.csv")

# Use .columns to view the column names
print(f"The columns in the raw dataset are:\n\n{df.columns}")

The columns in the raw dataset are:

Index(['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'],
      dtype='object')


### **1.3 Drop Two End Columns**

In the write-up on Kaggle for this dataset, the author requested we delete the last two columns in the dataset:

* *"PLEASE IGNORE THE LAST 2 COLUMNS (NAIVE BAYES CLAS…). I SUGGEST TO RATHER DELETE IT BEFORE DOING ANYTHING"*.

As I am already using `.gitignore` for this raw dataset to prevent the upload of PII to GitHub, I am going to remove these two columns in this first notebook.

In [6]:
# Drop the specified columns using df.drop

df = df.drop(columns=["Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1", "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2"])

# Call df.columns to verify that the columns have been dropped
df.columns

Index(['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
      dtype='object')

In [7]:
# Show the shape of the DataFrame to establih the number of rows and columns in the dataset
print(f"The shape of the DataFrame is: {df.shape}")

The shape of the DataFrame is: (10127, 21)


With the two end columns removed, we are working with a DataFrame that has 10,127 rows and 21 columns.

---

## **2. Set up Salting and Hashing**

### **2.1 Salting**

The salt is added to each `CLIENTNUM` before we encode the data using hashing. This is done in order to complicate each ID before being hashed.
* I have called the variable `DONOR_SALT` and this is stored in a `.env` file.

#### **2.1.1 Create a .env**

Having consulted blog posts/ articles on Medium.com and using Generative AI (Claude Sonnet 5), I discovered that the salt is often placed in a `.env` file. In order to load the `DONOR_SALT` from the `.env`, I needed to load a new package - `python-dotenv==1.2.2`. This has been added to my `requirements.txt` file.

I have made sure to exclude this `.env` from my commits by putting it in `.gitignore` (this will ensure that the `DONOR_SALT` is not shared and is kept secure).

In [8]:
# Load variables from .env file into the project environment
load_dotenv()

True

When running this code, we get the output `True`. This means that the project envrionment has successfully loaded.

#### **2.1.2 Assign DONOR_SALT to a variable to use in the process of anonymising CLIENTNUM**

Now that the `.env` had been loaded into the project, I need to assign `DONOR_SALT` to a variable.
* I have selected a variable, `SALT`.

> *Notes on Process*: 
<br><br>I used Microsoft Copilot's inline suggestions to support me in writing these lines of code, particularly the first line where I needed to use `os.getenv`.
<br><br>

In [9]:
# Assign the salt value from the environment variable to a variable
SALT = os.getenv("DONOR_SALT")

# Check that this loaded correctly by printing a message if SALT was loaded correctly
if SALT:
    print("The SALT variable has loaded correctly.")
else:
    print("The SALT variable has not loaded. Please check your .env file.")

The SALT variable has loaded correctly.


### **2.2 Salting and Hashing**

In this next section, I need to combine the client number values with the sal and then hash the combination.

#### **2.2.1 Check Data Type of Client Number values**

I learned that environment variables in `.env` files are read as strings, therefore I need to make sure that the values in the `CLIENTNUM` column are also strings.

In [10]:
# Check dtype of CLIENTNUM column
print(f"Data Type of CLIENTNUM column: {df['CLIENTNUM'].dtype}")

Data Type of CLIENTNUM column: int64


Because this code says that the datatype is an integer, I will need to change the type of all the values in this column to strings.

#### **2.2.2 Create Salt and Hash Function**

In the imported libraries and packages, I have already imported `hashlib` - within this library I am going to use SHA-256 to hash my client numbers.
* In order to do this, I learned that the string we create (made up of the client numbers plus the SALT) *must be coverted to bytes*.

In [11]:
# Create a function that we .apply to the values of the donor_id column to combine with SALT and hash the result
def salt_hash_clientnum(clientnum):
    # Combine the donor_id and SALT into one string and conver it to bytes
    salted_clientnum = (str(clientnum) + SALT).encode("utf-8")
    # Hash the salted donor_id using SHA-256
    hashed_clientnum = hashlib.sha256(salted_clientnum).hexdigest()
    return hashed_clientnum

To test that the function has worked as expected, I have created a test where I call the newly-created function.

In [12]:
#Example usage of the function
print(salt_hash_clientnum("donor42"))

print(salt_hash_clientnum(987654321))

1b2f136682f1c47472b11c97c44af355baf639723661e11d708a6370db76fcb5
dd92651008573641ea13ef7546d468bf58ec8d62cf5937b6023841f9813beff1


Because these two example outputs are hexadecimal strings with the characters 0-9 and a-f, I can be confident that this function has worked as expected.

---

## **3. Update DataFrame with Anonymised Client Numbers**

### **3.1 Apply Salting and Hashing Function**

I will now create an anonymised column in the DataFrame by applying our `salt_hash_clientnum` function and check its contents to ensure that the function has worked as expected on the column.

In [13]:
# Create anonymised column in the raw DataFrame
df["anonymised_clientnum"] = df["CLIENTNUM"].apply(salt_hash_clientnum)

# Check that the new column has been created and the new values have been hashed
print(f"Contents of 'anonymised_clientnum' column:\n\n{df['anonymised_clientnum'].head(5)}")

Contents of 'anonymised_clientnum' column:

0    022e52f0a251431f954db620fd0c87ac3c523c60cc5980...
1    2731de4ed9ecb2a3ab828448bfda6137e5c7571e1f7576...
2    75dac624c2bbdb15b16abc0350820bcf598c012a76b102...
3    1aaada0cd1f7dc23d83a171beec9f398cfac1471841843...
4    1811f55b3210153f77e41ceceb61cc181d2edc4e65947e...
Name: anonymised_clientnum, dtype: object


In [15]:
# Call df.columns to see the column names and to check that the new column has been created
print(f"Column names:\n\n{df.columns}")

Column names:

Index(['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'anonymised_clientnum'],
      dtype='object')


### **3.2 Delete CLIENTNUM Column**

Now that we have the new anonymised version of the client numbers, I am safe to delete the original `CLIENTNUM` column from my DataFrame.

In [16]:
# Drop the "CLIENTNUM" column from the DataFrame
df = df.drop(labels=["CLIENTNUM"], axis=1)

# Confirm the column deletion by printing the DataFrame columns
print("DataFrame Columns after dropping 'CLIENTNUM' column:\n", df.columns)

DataFrame Columns after dropping 'CLIENTNUM' column:
 Index(['Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count',
       'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category',
       'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'anonymised_clientnum'],
      dtype='object')


### **3.3 Change Column Order**

Finally, to clean up my DataFrame, I want to move the `anonymised_clientnum` column from the end of the DataFrame to the beginning. 

This was a new skill that I had only practiced once before and I relied on an article written via Medium.com to get the correct code. The author and the article are listed in the **Credits** section of the **README.md**.

In [17]:
# Update column order to have the "anonymised_clientnum" column first, followed by the other columns
new_column_order = ["anonymised_clientnum"] + [col for col in df.columns if col != "anonymised_clientnum"]
df = df[new_column_order]
df.columns

Index(['anonymised_clientnum', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
      dtype='object')

In [18]:
# Show the shape of the DataFrame to know the number of rows and columns after the anonymisation and column deletion
print(f"Shape of the DataFrame: {df.shape}")

Shape of the DataFrame: (10127, 21)


We can see that the number of columns is once again 21, demonstrating we have replaced the original `CLIENTNUM` with the new `anonymised_clientnum`.

---